![interpreto_banner](../assets/img/interpreto_banner.png){ style="display:block; max-width:100%; height:auto; margin:0 auto;" }

# Classification Concept-based Explanation Tutorial

Welcome to this tutorial, our will be to obtain concept-based explanations starting from the beginning.

For any precision, please refer to the [**Interpreto documentation**](https://for-sight-ai.github.io/interpreto/).

There are five key steps for concepts based explanations:

1. [➗ **Split** your model in two parts](#split)
2. [🚦 Compute a dataset of **activations**](#activations)
3. [🏋️‍♂️ **Fit** a concept model on activations](#fit)
4. [🏷️ **Interpret** the concept dimensions](#interpret)
5. [🌍 Find the globally **important** concepts](#important)

On which we add three bonus steps:

6. [📚 **Class-wise** concepts and LLM label](#class-wise)
7. [📍 **Locally** important concepts](#locally)
8. [⚖️ **Evaluate** concept-based explanations](#evaluate)

*Author: Antonin Poché*

In [1]:
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. ➗ **Split** your model in two parts <a class="anchor" id="split"></a>

We choose a `DistilBERT` fine-tuned on the `AG-News` dataset and split it just before the classification head.

To split the model, we use [`interpreto.SplitterForClassification`](https://for-sight-ai.github.io/interpreto/api/concepts/splitters/splitter_for_classification/), which wraps the `transformers` model.

It splits the model at the [CLS] token, thus the first part is the encoder and the second the classification head. Which is the setup we highly recommend for interpretability.

For token-level or pooled representations from causal and encoder text models, use [`interpreto.TextTokensSplitter`](https://for-sight-ai.github.io/interpreto/api/concepts/splitters/text_tokens_splitter/) instead.

In [2]:
from interpreto import SplitterForClassification

splitter = SplitterForClassification(
    model_or_repo_id="textattack/distilbert-base-uncased-ag-news",
    device_map="cuda",
    batch_size=64,
)

## 2. 🚦 Compute a datasets of **activations** <a class="anchor" id="activations"></a>

We load the `AG-News` train set.

Then we extract the activations of the [CLS] token of each document.

[`interpreto.SplitterForClassification.get_activations()`](https://for-sight-ai.github.io/interpreto/api/concepts/splitters/splitter_for_classification/#interpreto.SplitterForClassification.get_activations)

In [3]:
from datasets import load_dataset

# load the AG-News dataset
dataset = load_dataset("fancyzhx/ag_news")
inputs = list(dataset["train"]["text"])
classes_names = dataset["train"].features["label"].names

# Compute the [CLS] token activations
activations, predictions = splitter.get_activations(inputs, tqdm_bar=True)

  0%|          | 0/1875 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

100%|██████████| 1875/1875 [01:10<00:00, 26.78it/s]


## 3. 🏋️‍♂️ **Fit** a concept model on activations <a class="anchor" id="fit"></a>

With activations, we can train a concept model to find patterns (concepts).

The `concept_model` is an attribute of our concept explainer, alongside the `splitter`. With these two elements, we can go from inputs to concepts and from concepts to outputs.

In this tutorial, we use [`interpreto.concepts.ICAConcepts`](https://for-sight-ai.github.io/interpreto/api/concepts/concept_spaces/optim/#interpreto.concepts.ICAConcepts) built upon the ICA (Independent Component Analysis) dimension reduction algorithm.

There are at least 15 others concept model available in interpreto. do not hesitate to explore them.

> 🔥 **Tip**
>
> `ICAConcepts` is a good first candidate for classification. It has no requirements, is fast, and provide correct first results on most datasets.
>
> Well the `SemiNMFConcepts` used in the [better concepts section](#class-wise) is too.

In [4]:
from interpreto.concepts import ICAConcepts

# instantiate the concept explainer
concept_explainer = ICAConcepts(splitter, nb_concepts=50, device="cuda")

# fit the concept explainer on activations
concept_explainer.fit(activations)

## 4. 🏷️ **Interpret** the concept dimensions <a class="anchor" id="interpret"></a>

We have our concepts and the link between concepts and classes. But now, we need to make sense of these concepts.

In this case, we will use the [`interpreto.concepts.interpretations.TopKInputs`](https://for-sight-ai.github.io/interpreto/api/concepts/interpretations/topk_inputs/#interpreto.concepts.interpretations.TopKInputs) to find the 8 words which activates the most our concepts.

In [5]:
from interpreto.concepts.interpretations import TopKInputs

# instantiate the interpretation method with the concept explainer
topk_inputs_method = TopKInputs(
    concept_explainer=concept_explainer,
    k=5,
    use_unique_words=3,  # for classification we are force to use unique words or ngrams
    unique_words_kwargs={
        "count_min_threshold": round(
            len(inputs) * 0.002
        ),  # appear in at least 0.2% of the samples | increase if random words appear and decrease if some words appear too often
        "lemmatize": True,
        "words_to_ignore": [],  # include noise words and punctuation
    },
)

In [6]:
# call the interpretation methods on the inputs
# we cannot give the previously computed activations because `use_unique_words=True` creates samples with a single word inside
topk_words = topk_inputs_method.interpret(
    inputs=inputs,
    concepts_indices="all",
)

## 5. 🌍 Find the globally **important** concepts <a class="anchor" id="important"></a>

We have concept directions, it means that our model has access to them, but not that it uses them.

It is the same when you train a model on tabular data, not all features are used.

In this step, we use the [`ConceptAutoEncoderExplainer.concept_output_gradients`](https://for-sight-ai.github.io/interpreto/api/concepts/concept_spaces/base/#interpreto.concepts.ConceptAutoEncoderExplainer.concept_output_gradient) to evaluate the importance of each concept with respect to the predicted classes.

> ➡️ **Note**
>
> All unsupervised concept-based explainers in Interpreto inherit from [`ConceptAutoEncoderExplainer`](https://for-sight-ai.github.io/interpreto/api/concepts/concept_spaces/base/#interpreto.concepts.ConceptAutoEncoderExplainer).

> ➡️ **Note 2**
>
> This step can be done prior to the interpretation, as the interpretation step can be compute heavy. Then specify using the `concept_indices` parameter.
> Only interpreting the important concepts can be wise. (Here we only have 50 concepts, so it does not matter.)

In [7]:
import torch

# estimate the importance of concepts for each class using the gradient
gradients = concept_explainer.concept_output_gradient(
    inputs=activations,
    targets=None,  # None means all classes
    tqdm_bar=True,
)

# stack gradients on samples and average them over samples
mean_gradients = torch.stack(gradients).abs().squeeze().mean(0)  # (num_classes, num_concepts)

# for each class, sort the importance scores
order = torch.argsort(mean_gradients, descending=True)

# visualize the top 5 concepts for each class
for target in range(order.shape[0]):
    print(f"\nClass: {classes_names[target]}:")
    for i in range(5):
        concept_id = order[target, i].item()
        importance = mean_gradients[target, concept_id].item()
        words = list(topk_words.get(concept_id, None).keys())
        print(f"\tconcept id: {concept_id},\timportance: {round(importance, 3)},\ttopk words: {words}")

100%|██████████| 1875/1875 [00:05<00:00, 321.25it/s]



Class: World:
	concept id: 17,	importance: 0.113,	topk words: ['hamid karzai', 'fallujah', 'of baghdad', 'baghdad', 'baghdad ,']
	concept id: 8,	importance: 0.066,	topk words: ['the un', 'un', 'china # 39', 'security council', 'convention']
	concept id: 5,	importance: 0.061,	topk words: ['democratic', 'cp ) -', 'john kerry', 'opposition', 'lt ; p']
	concept id: 12,	importance: 0.05,	topk words: ['blog', 'pirate', 'newsfactor -', 'washingtonpost.com', 'http']
	concept id: 47,	importance: 0.043,	topk words: ['kidnapped', 'kidnapper', 'hostage in', '( canadian press', 'hostage']

Class: Sports:
	concept id: 17,	importance: 0.047,	topk words: ['hamid karzai', 'fallujah', 'of baghdad', 'baghdad', 'baghdad ,']
	concept id: 43,	importance: 0.043,	topk words: ['ap -', 'ap - the', 'ap ) ap', 'ap - a', ') ap -']
	concept id: 39,	importance: 0.04,	topk words: ['linux', 'supercomputer', 'cisco', 'java', 'ibm']
	concept id: 15,	importance: 0.037,	topk words: ['gt ; on', 'gt ;', 'gt ; the', 'calif.

In [8]:
from interpreto import plot_concepts

labels = {k: list(v.keys()) for k, v in topk_words.items()}

plot_concepts(
    classes_names=classes_names,
    concepts_importances=mean_gradients,
    concepts_labels=labels,
)

> ❓ **The concepts are not interpretable, what do I do?**
>
> - Try to improve the concept-space:
>   - Increases the number of samples. You can artificially do so by splitting then by sentences (not included)
>   - Try different concept-models and parameters
>   - Try to compute concepts class-wise see [next section](#class-wise)
>
> - Improve the interpretation of concepts:
>   - Play with the parameters
>   - Try [`LLMLabels`](https://for-sight-ai.github.io/interpreto/api/concepts/interpretations/llm_labels/#interpreto.concepts.interpretations.LLMLabels) see [next section](#class-wise)
>
> - Try to evaluate the concepts to automatically find the best methods.
>
> - Never forget the **faithfulness-plausibility trade-off** of explanations

## 6. 📚 Better concepts with class-wise concepts and LLM labels <a class="anchor" id="class-wise"></a>

This section aims at improving the concepts learned by the model. We try three different approaches:

- Training class-wise concepts
- Using another concept model: [`SemiNMFConcepts`](https://for-sight-ai.github.io/interpreto/api/concepts/concept_spaces/optim/#interpreto.concepts.SemiNMFConcepts)
- Using [`interpreto.concepts.LLMLabels`](https://for-sight-ai.github.io/interpreto/api/concepts/interpretations/llm_labels/#interpreto.concepts.interpretations.LLMLabels) to interpret the concepts

When a single concept-space is defined for all classes, concepts tend to correspond to the classes themselves. In particular, when the concept-space is built upon on the latent space just before the classification head.

In this section, we will learn a concept space for each class separately. Thus, the class-wise concept explainers will only see examples from a single class (based on the predictions).

> ℹ️ **Note**
>
> `LLMLabels` loads Qwen3-0.6B from the Hugging Face Hub to label the concepts locally.

In [ ]:
from interpreto.commons.llm_interface import HuggingFaceLLM
from interpreto.concepts import LLMLabels, SemiNMFConcepts

concept_explainers = {}
concept_interpretations = {}
concept_importances = {}

# We could pass the repo id directly, but do it here to prevent creating several instances of the model
llm_interface = HuggingFaceLLM(
    "Qwen/Qwen3.5-9B",
    device=DEVICE,
    batch_size=4,
    generation_kwargs={"max_new_tokens": 200},
    chat_template_kwargs={"enable_thinking": False},
)

# iterate over classes
for target, class_name in enumerate(classes_names):
    # ----------------------------------------------------------------------------------------------
    # 2. construct the dataset of activations (extract the ones related to the class)
    indices = (predictions == target).nonzero(as_tuple=True)[0]
    if not len(indices):
        continue
    class_wise_inputs = [inputs[i] for i in indices]
    class_wise_activations = activations[indices]

    # ----------------------------------------------------------------------------------------------
    # 3. train concept model
    concept_explainers[target] = SemiNMFConcepts(splitter, nb_concepts=20, device="cuda")
    concept_explainers[target].fit(class_wise_activations)

    # ----------------------------------------------------------------------------------------------
    # 5. compute concepts importance (before interpretations to limit the number of concepts interpreted)
    gradients = concept_explainers[target].concept_output_gradient(
        inputs=class_wise_activations,
        targets=[target],
        concepts_x_gradients=True,
        batch_size=64,
    )

    # stack gradients on samples and average them over samples
    concept_importances[target] = torch.stack(gradients, axis=0).squeeze().abs().mean(dim=0)  # (num_concepts,)

    # for each class, sort the importance scores
    important_concept_indices = torch.argsort(concept_importances[target], descending=True).tolist()

    # ----------------------------------------------------------------------------------------------
    # 4. interpret the important concepts concepts
    llm_labels_method = LLMLabels(
        concept_explainer=concept_explainers[target], llm_interface=llm_interface, k_examples=20
    )

    concept_interpretations[target] = llm_labels_method.interpret(
        inputs=class_wise_inputs,
        concepts_indices="all",  # we could pass `important_concept_indices[:5]` but we later need more for local explanations
    )

    print(f"\nClass: {class_name}")
    for concept_id in important_concept_indices[:5]:
        label = concept_interpretations[target].get(concept_id, None)
        importance = concept_importances[target][concept_id].item()
        if label is not None:
            print(f"\timportance: {round(importance, 3)},\t{label}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]


Class: World
	importance: 0.088,	International Conflict and Peace Processes
	importance: 0.076,	Election Disputes and Controversies
	importance: 0.074,	Global Terrorist and Aviation Incidents
	importance: 0.074,	Military and Political Conflict
	importance: 0.073,	U.S. News and Politics

Class: Sports
	importance: 0.08,	College sports news
	importance: 0.073,	Olympic and Professional Sports News
	importance: 0.072,	Sports news headlines
	importance: 0.068,	Football player disciplinary actions
	importance: 0.067,	Major League Baseball Game Summaries

Class: Business
	importance: 0.096,	Regulatory actions on pharmaceuticals
	importance: 0.082,	Corporate earnings and financial news
	importance: 0.074,	Corporate Governance and Financial Regulation
	importance: 0.073,	Global Economic Indicators
	importance: 0.071,	Corporate governance and regulatory actions

Class: Sci/Tech
	importance: 0.077,	Digital Media and Internet Culture
	importance: 0.076,	Mobile phone industry news
	importance: 0.0

In [12]:
plot_concepts(
    classes_names=classes_names,
    concepts_importances=concept_importances,
    concepts_labels=concept_interpretations,
)

## 6. 📍 **Locally** important concepts <a class="anchor" id="locally"></a>

### 6.1 Local concepts-to-ouputs attributions

We got which concept are important for the classes globally. However, the concepts are not all present in each sample and the model might rely on a specific concept for a specific sample. Let's look at locally important concepts, meaning, the concepts the model used in a specific sample.

> ➡️ Note
>
> We cannot look at which word activates which concept without doing a forward pass for each word individually, because we use the [CLS] token. You could do the following cell, iterate on words and replace the example by the word.

In [13]:
example = "The Iranian basketball team won against the Iraqi team in the final match."

local_activations, local_predictions = splitter.get_activations([example])

pred = local_predictions.item()
concepts_activations = concept_explainers[pred].activations_to_concepts(local_activations)

print(f"Example: {example}")
print(f"Predicted class: {classes_names[pred]}")

# compute local concepts importance for the class
# we use the class-wise
local_importance = concept_explainers[pred].concept_output_gradient(
    inputs=[example],
    concepts_x_gradients=True,
)[0]  # there is only one sample

plot_concepts(
    # sample=[example],
    classes_names=classes_names,
    # concepts_activations=concepts_activations,
    concepts_importances=local_importance.squeeze(),  # importance of shape (t, g, c) -> (t, c)
    concepts_labels=concept_interpretations,
)

Example: The Iranian basketball team won against the Iraqi team in the final match.
Predicted class: Sports


### 6.2 Inputs-to-concepts attributions

This can be seen as an interpretation, but it works even better when concepts have labels as defined earlier. (Note that these concepts are not absolute.)

To obtain inputs-to-concepts attributions, we use the `get_inputs_to_concepts_model()` method from concepts explainers and give it to perturbation-based attribution methods.

In [14]:
from interpreto import KernelShap

attributions = {}
# Iterate on explainers because we did class-wise explanations
for class_id, cpt_explainer in concept_explainers.items():
    attribution_explainer = KernelShap(
        model=cpt_explainer.get_inputs_to_concepts_model(),
        tokenizer=splitter.tokenizer,
    )

    # Compute the attributions
    attribution_outputs = attribution_explainer.explain(
        example,
        targets=None,  # we want to explain all concepts
    )[0]
    attributions[class_id] = attribution_outputs.attributions.T

# Visualization
plot_concepts(
    sample=attribution_outputs.elements,
    classes_names=classes_names,  # TODO WARNING THE CLASS IS NOT CORRECT
    concepts_activations=attributions,  # argument name is miss-leading because we use concepts activations for generation
    concepts_importances=local_importance.squeeze(),  # importance of shape (t, g, c) -> (t, c)
    concepts_labels=concept_interpretations,
)

## 7. ⚖️ **Evaluate** concept-based explanations <a class="anchor" id="evaluate"></a>

We take back the `ICAConcepts` explainer and evaluate it on new samples.

In [15]:
test_inputs = dataset["test"]["text"][:1000]  # let's take one thousand test samples
test_labels = torch.tensor(dataset["test"]["label"][:1000])

# Compute the [CLS] token activations
test_activations, test_predictions = splitter.get_activations(inputs=test_inputs)

### 7.1 🌐 Evaluate the concept-space from the [third part](#fit)

> ⚠️ Warning:
>
> These metrics should only be used to compare the concept-space trained in similar contexts, same model, split point, activation dataset...

#### Reconstruction error

- [`interpreto.concepts.metrics.MSE`](https://for-sight-ai.github.io/interpreto/api/concepts/metrics/reconstruction_metrics/#interpreto.concepts.metrics.MSE)
- [`interpreto.concepts.metrics.FID`](https://for-sight-ai.github.io/interpreto/api/concepts/metrics/reconstruction_metrics/#interpreto.concepts.metrics.FID)

In [16]:
from interpreto.concepts.metrics import FID, MSE

mse = MSE(concept_explainer).compute(test_activations)
fid = FID(concept_explainer).compute(test_activations)

print(f"MSE: {round(mse, 3)}, FID: {round(fid, 3)}")

MSE: 64.51, FID: 0.024


> ➡️ Note
>
> Alone these values are useless, they should be compared between several concept explainers.

#### Sparsity

- [`interpreto.concepts.metrics.Sparsity`](https://for-sight-ai.github.io/interpreto/api/concepts/metrics/sparsity_metrics/#interpreto.concepts.metrics.Sparsity)
- [`interpreto.concepts.metrics.SparsityRatio`](https://for-sight-ai.github.io/interpreto/api/concepts/metrics/sparsity_metrics/#interpreto.concepts.metrics.SparsityRatio)

In [17]:
from interpreto.concepts.metrics import Sparsity, SparsityRatio

sparsity = Sparsity(concept_explainer).compute(test_activations)
ratio = SparsityRatio(concept_explainer).compute(test_activations)

print(f"Sparsity: {round(sparsity, 3)}, Sparsity ratio: {round(ratio, 3)}")

Sparsity: 1.0, Sparsity ratio: 0.02


#### Dictionary metrics

- [`interpreto.concepts.metrics.Stability`](https://for-sight-ai.github.io/interpreto/api/concepts/metrics/dictionary_metrics/#interpreto.concepts.metrics.Stability)

The `Stability` metric requires two concept explainer, hence our first step step will be to train a new `ICAConcepts` with the same model, split, dataset, and hyper-parameters as the original `ICAConcepts`. However, to get a statistically robust metric score, one should compare more than just two instances of the same explainer.

In [18]:
from interpreto.concepts.metrics import Stability

# instantiate and train a second concept explainer
second_explainer = ICAConcepts(splitter, nb_concepts=50, device="cuda")
second_explainer.fit(activations)

stability = Stability(concept_explainer, second_explainer).compute()
del second_explainer

print(f"Stability: {round(stability, 3)}")

Stability: 1.0


### 7.2 💭 Evaluate the concepts-interpretations from the [fourth step](#important)

In [19]:
# Work in progress, coming soon

### 7.3 ↔️ Evaluate the whole concept-based explanations with `ConSim`

`ConSim` is a metric evaluating the whole concept-based explanations in an end-to-end manner. Indeed, this metric, evaluates to which extend the provided concept-based explanations help a meta-predictor to predict what the studied model would have predicted. The idea is that is a meta-predictor understands the model, it is able to predict what the model would have predicted on new samples.

- [`interpreto.concepts.metrics.ConSim`](https://for-sight-ai.github.io/interpreto/api/concepts/metrics/consim/#interpreto.concepts.metrics.ConSim)

> ➡️ Note
>
> For significant scores, we iterate on 10 different seeds. (5 were used in the paper).

In [20]:
from interpreto.commons.llm_interface import HuggingFaceLLM
from interpreto.concepts.metrics.consim import ConSim, PromptTypes

# convert step 5 global concept importances to a dictionary
global_importances = {
    class_name: dict(enumerate(importances))
    for class_name, importances in zip(classes_names, mean_gradients, strict=True)
}

# Initialize the ConSim with the split model and the user LLM
# Therefore, a given ConSim metric can be used on different explainers for cleaner comparison
con_sim = ConSim(
    splitter,
    user_llm=llm_labels_method.llm_interface,  # Reuse the local llm used for concepts labeling (equivalent to "Qwen/Qwen3-0.6B")
    classes=classes_names,
)

baseline_list = []
ica_score_list = []
for seed in range(10):
    # Select examples for evaluation
    samples, labels, predictions = con_sim._extract_interesting_elements(
        inputs=test_inputs,
        labels=test_labels,
        predictions=test_predictions,
        seed=seed,
    )

    # Compute a baseline and ConSim score to give sense to the explainer ConSim score
    baseline = con_sim.evaluate(
        interesting_samples=samples, predictions=predictions, prompt_type=PromptTypes.L2_baseline_with_lp
    )

    if baseline is None:
        continue

    # Compute the ConSim score for an explainer
    ica_score = con_sim.evaluate(
        interesting_samples=samples,
        predictions=predictions,
        concept_explainer=concept_explainer,
        concepts_interpretation=topk_words,
        global_importances=global_importances,
        prompt_type=PromptTypes.E2_global_concepts_with_lp,
    )

    if ica_score is None:
        continue

    baseline_list.append(baseline)
    ica_score_list.append(ica_score)

print(f"Baseline: {round(sum(baseline_list) / 10, 2)}, ICA: {round(sum(ica_score_list) / 10, 2)}")

/home/antonin.poche/interpreto/interpreto/concepts/metrics/consim.py:552: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:820.)
  importances = torch.abs(torch.Tensor(list(concepts_importance.values())))


Baseline: 0.28, ICA: 0.28


> ➡️ Note
>
> We evaluated the first ICA explainer here. Concepts where not really interpretable, ConSim agrees.

## 8. Using your own LLM interface <a class="anchor" id="interface"></a>

In [ ]:
from interpreto.commons.llm_interface import LLMInterface


class GeminiLLM(LLMInterface):
    def __init__(self, api_key: str, model: str = "gemini-1.5-flash"):
        try:
            import google.generativeai as genai  # noqa: PLC0415  # ruff: disable=import-outside-toplevel
        except ImportError as e:
            raise ImportError("Install google-generativeai to use Google Gemini API.") from e

        self.genai = genai
        self.genai.configure(api_key=api_key)
        self.model = model

    def generate(self, system_prompt: str, user_prompt: str, **generation_kwargs) -> str:
        model = self.genai.GenerativeModel(
            model_name=self.model,
            system_instruction=system_prompt,
        )
        response = model.generate_content(user_prompt, **generation_kwargs)
        return response.text.strip()